##Importing Functions

In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.functions import length,trim
print(' functions imported succesfully')

##reading the table

In [0]:
df=spark.read.table('workspace.bronze.pizza_types')
print('table read succesfully!')

##bronze Layer validation

In [0]:
print('check schema...')
df.printSchema()
print('check data types...')
print(df.dtypes)
print('check data sample...')
df.show(3)

##column operations

In [0]:
#naming convention is snake_case no need to change
#no bad data types but change as defensive approach
df=df.withColumn('pizza_type_id',col('pizza_type_id').cast('string')).withColumn('name',col('name').cast('string')).withColumn('category',col('category').cast('string')).withColumn('ingredients',col('ingredients').cast('string'))
print('data types changed succesfully!')

#check for trailling spaces,trimming
def trim_spaces(dataframe):
    for colname,coltype in dataframe.dtypes:
        if coltype=='string':
            count=dataframe.filter(length(trim(col(colname)))!=length(col(colname))).count()
            if count>0: 
                print(f'column {colname} contains {count} trailling spaces')
                dataframe=dataframe.withColumn(colname,trim(col(colname)))
                print(f'{colname} trimmed successfully!')
            else :
                print('checked successfully!')
                print(f'no trailling spaces found in {colname}')
            
    return dataframe
    
df=trim_spaces(df)

##row operations

In [0]:
#duplicates check
df_unique_count=df.distinct().count()
df_original_count=df.count()
if df_original_count != df_unique_count:
    print('duplicates found!')
    df=df.distinct()
    print('duplicates handled successfully')
else :
    print('no duplicates found!')
    print('checked successfully!')

print('-'*40)
#nulls check
df_anynulls_count=df.na.drop('any').count()
df_allnulls_count=df.na.drop('all').count()

if df_allnulls_count != df_original_count or df_anynulls_count != df_original_count:
    print('nulls found!')
    print('original row count',df_original_count)
    print('rows count after removing rows containing ANY nulls',df_anynulls_count)
    print('rows count after removing rows containing ALL nulls',df_allnulls_count)
    raise ValueError('Null values detected in dataset. Pipeline halted.')
else :
    print('no nulls found!')
    print('checked successfully!')






##Validation

In [0]:
print('-'*40)
print('Data validation checks')
print('-'*40)
validation_results=[]
def validate(colname,rule,condition) :
    invalid= df.filter(condition).count()
    validation_results.append({
        'column' : colname,
        'rule' : rule,
        'invalid_count' : invalid,
        'invalid_pct': round(invalid / df.count() * 100, 2)
    })


valid_categories = [row[0] for row in df.select('category').distinct().collect()]
validate("category", "invalid category", ~col("category").isin(valid_categories))

spark.createDataFrame(validation_results).show(truncate=False)



##saving

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.pizza_types')
print('saved successfully!')

In [0]:
%sql
--sanity check
SELECT * FROM workspace.silver.pizza_types LIMIT 10